# AML Transaction Monitoring Model
**Author:** Anushka Shinde | MS Finance, Boston University  




In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

customer_types = ['Individual', 'Business', 'Shell Company', 'NGO']
countries = ['USA', 'UK', 'India', 'Cayman Islands', 'Panama',
             'Switzerland', 'Germany', 'UAE', 'Nigeria', 'Singapore']
high_risk_countries = ['Cayman Islands', 'Panama', 'Nigeria']
transaction_types = ['Wire Transfer', 'Cash Deposit', 'ATM Withdrawal',
                     'Online Transfer', 'Check', 'Crypto Exchange']
n = 500

df = pd.DataFrame({
    'Transaction_ID': [f'TXN{str(i).zfill(5)}' for i in range(1, n + 1)],
    'Customer_ID': [f'CUST{np.random.randint(1000, 2000)}' for _ in range(n)],
    'Customer_Type': np.random.choice(customer_types, n, p=[0.5, 0.3, 0.1, 0.1]),
    'Transaction_Amount': np.round(
        np.where(
            np.random.rand(n) > 0.95,
            np.random.uniform(9000, 9999, n),
            np.random.exponential(scale=3000, size=n).clip(100, 100000)
        ), 2),
    'Transaction_Type': np.random.choice(transaction_types, n),
    'Origin_Country': np.random.choice(countries, n,
        p=[0.3, 0.15, 0.15, 0.05, 0.05, 0.08, 0.1, 0.05, 0.04, 0.03]),
    'Destination_Country': np.random.choice(countries, n),
    'Num_Transactions_Last_30Days': np.random.randint(1, 50, n),
    'Avg_Transaction_Last_6Months': np.round(
        np.random.exponential(scale=2000, size=n).clip(100, 50000), 2),
    'Account_Age_Years': np.round(np.random.uniform(0.1, 20, n), 1),
    'Prior_SAR_Filed': np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

def apply_red_flags(row):
    flags = []
    if 9000 <= row['Transaction_Amount'] <= 9999:
        flags.append('Structuring')
    if row['Origin_Country'] in high_risk_countries or \
       row['Destination_Country'] in high_risk_countries:
        flags.append('High-Risk Country')
    if row['Avg_Transaction_Last_6Months'] > 0:
        if row['Transaction_Amount'] / row['Avg_Transaction_Last_6Months'] > 5:
            flags.append('Unusual Amount vs History')
    if row['Customer_Type'] in ['Shell Company', 'NGO'] and \
       row['Transaction_Type'] == 'Wire Transfer':
        flags.append('High-Risk Entity Type')
    if row['Prior_SAR_Filed'] == 1:
        flags.append('Prior SAR on File')
    if row['Account_Age_Years'] < 1 and row['Transaction_Amount'] > 10000:
        flags.append('New Account High Value')
    if row['Num_Transactions_Last_30Days'] > 30:
        flags.append('Excessive Transaction Frequency')
    return '; '.join(flags) if flags else 'None'

df['Red_Flags'] = df.apply(apply_red_flags, axis=1)
df['Flag_Count'] = df['Red_Flags'].apply(
    lambda x: 0 if x == 'None' else len(x.split(';')))

print(f'Stages 1 & 2 complete — {len(df)} transactions with red flags ready.')

Stages 1 & 2 complete — 500 transactions with red flags ready.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print('ML libraries loaded!')

ML libraries loaded!


In [ ]:
features = [
    'Transaction_Amount',
    'Num_Transactions_Last_30Days',
    'Avg_Transaction_Last_6Months',
    'Account_Age_Years',
    'Prior_SAR_Filed',
    'Flag_Count'
]

# Extract just these columns into a new variable X
# X is the standard name for input data in ML
X = df[features]

print('Features selected!')
print(f'   Shape of X: {X.shape}  (500 rows, 6 features)')
print('\nPreview of feature data:')
X.head()

Features selected!
   Shape of X: (500, 6)  (500 rows, 6 features)

Preview of feature data:


,Transaction_Amount,Num_Transactions_Last_30Days,Avg_Transaction_Last_6Months,Account_Age_Years,Prior_SAR_Filed,Flag_Count
0,1392.82,30,141.81,12.7,0,1
1,927.13,46,1081.43,15.7,1,3
2,9325.60,20,247.41,12.0,0,3
3,2075.45,39,1897.61,8.3,0,1
4,927.82,12,5576.86,19.2,0,0


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Features normalized!')
print(f'   Shape after scaling: {X_scaled.shape}')
print(f'\n   Example — Transaction_Amount before scaling: ${df["Transaction_Amount"].iloc[0]:,.2f}')
print(f'   Example — Transaction_Amount after scaling : {X_scaled[0][0]:.4f}')

Features normalized!
   Shape after scaling: (500, 6)

   Example — Transaction_Amount before scaling: $1,392.82
   Example — Transaction_Amount after scaling : -0.6141


In [ ]:
# Initialize the model
model = IsolationForest(
    contamination=0.08,   # We expect ~8% of transactions to be anomalous
    random_state=42       # For reproducibility
)

# Train the model on our scaled data
# This is where the model learns what 'normal' looks like
model.fit(X_scaled)

print('Isolation Forest model trained!')

Isolation Forest model trained!


In [ ]:
# Get raw anomaly scores from the model
raw_scores = model.score_samples(X_scaled)

print('Raw score examples (more negative = more suspicious):')
print(f'   Most normal score  : {raw_scores.max():.4f}')
print(f'   Most anomalous score: {raw_scores.min():.4f}')

# Convert to 0-100 scale
# Formula: flip and normalize so highest risk = 100
min_score = raw_scores.min()
max_score = raw_scores.max()

df['Risk_Score'] = ((raw_scores - max_score) / (min_score - max_score) * 100).round(1)

print(f'\nRisk scores calculated!')
print(f'   Score range: {df["Risk_Score"].min()} to {df["Risk_Score"].max()}')
print(f'   Average score: {df["Risk_Score"].mean():.1f}')

Raw score examples (more negative = more suspicious):
   Most normal score  : -0.3916
   Most anomalous score: -0.6841

Risk scores calculated!
   Score range: -0.0 to 100.0
   Average score: 27.7


In [ ]:
def assign_tier(score):
    if score >= 70:
        return 'HIGH'
    elif score >= 40:
        return 'MEDIUM'
    else:
        return 'LOW'

df['Risk_Tier'] = df['Risk_Score'].apply(assign_tier)

# Mark transactions that need immediate attention
df['Alert'] = df['Risk_Tier'].apply(lambda x: 'YES' if x == 'HIGH' else 'NO')

print('Risk tiers assigned!')
print('\n Risk Tier Breakdown:')
tier_counts = df['Risk_Tier'].value_counts()
for tier in ['HIGH', 'MEDIUM', 'LOW']:
    count = tier_counts.get(tier, 0)
    pct = round(count / len(df) * 100, 1)
    print(f'   {tier:<8} : {count:>4} transactions ({pct}%)')

Risk tiers assigned!

 Risk Tier Breakdown:
   HIGH     :   16 transactions (3.2%)
   MEDIUM   :   93 transactions (18.6%)
   LOW      :  391 transactions (78.2%)


In [ ]:
# Show the top 10 highest risk transactions
print('Top 10 Highest Risk Transactions:')
df.sort_values('Risk_Score', ascending=False)[
    ['Transaction_ID', 'Customer_Type', 'Transaction_Amount',
     'Origin_Country', 'Red_Flags', 'Risk_Score', 'Risk_Tier']
].head(10)

Top 10 Highest Risk Transactions:


,Transaction_ID,Customer_Type,Transaction_Amount,Origin_Country,Red_Flags,Risk_Score,Risk_Tier
468,TXN00469,Business,9617.43,Panama,Structuring; High-Risk Country; Unusual Amount...,100.0,HIGH
285,TXN00286,Shell Company,11814.61,Nigeria,High-Risk Country; Unusual Amount vs History; ...,98.1,HIGH
210,TXN00211,Business,11683.54,USA,None,92.3,HIGH
131,TXN00132,Business,6397.28,Singapore,High-Risk Country; Prior SAR on File,86.6,HIGH
407,TXN00408,Individual,5568.65,UK,None,85.1,HIGH
463,TXN00464,Individual,9074.56,USA,Structuring; Unusual Amount vs History; Prior ...,82.7,HIGH
117,TXN00118,Business,9097.30,USA,Structuring; Unusual Amount vs History; Prior ...,75.9,HIGH
225,TXN00226,NGO,8310.73,Cayman Islands,High-Risk Country; Prior SAR on File; Excessiv...,75.8,HIGH
71,TXN00072,Individual,401.40,Germany,Prior SAR on File,74.4,HIGH
1,TXN00002,Individual,927.13,Panama,High-Risk Country; Prior SAR on File; Excessiv...,73.5,HIGH


In [ ]:
# Compare average risk score by customer type
print('Average Risk Score by Customer Type:')
df.groupby('Customer_Type')['Risk_Score'].mean().round(1).sort_values(ascending=False)

Average Risk Score by Customer Type:


,Risk_Score
Customer_Type,
NGO,31.0
Shell Company,29.1
Business,28.2
Individual,26.5


In [ ]:
# Compare average risk score by transaction type
print('Average Risk Score by Transaction Type:')
df.groupby('Transaction_Type')['Risk_Score'].mean().round(1).sort_values(ascending=False)

Average Risk Score by Transaction Type:


,Risk_Score
Transaction_Type,
Wire Transfer,29.7
Cash Deposit,28.8
Crypto Exchange,28.7
Online Transfer,27.5
ATM Withdrawal,26.3
Check,25.8


In [ ]:
# Do flagged transactions actually get higher scores? They should!
print('Does the ML model agree with our rules?')
print('\nAverage Risk Score:')
print(f'   Transactions WITH red flags : {df[df["Flag_Count"] > 0]["Risk_Score"].mean():.1f}')
print(f'   Transactions WITHOUT flags  : {df[df["Flag_Count"] == 0]["Risk_Score"].mean():.1f}')
print('\nIf the ML score is higher for flagged transactions, our model is working correctly!')

Does the ML model agree with our rules?

Average Risk Score:
   Transactions WITH red flags : 27.8
   Transactions WITHOUT flags  : 27.4

If the ML score is higher for flagged transactions, our model is working correctly!
